# De tus videos a la trayectoria 3D

**Herramienta.** Pones tus videos en una carpeta y obtienes, con el mejor detector publicado en este repositorio, la
trayectoria de cada colibrí, sus 7 puntos y la lectura de profundidad. **Kernel:** `colibri-yolo`. No necesitas clave de
Roboflow ni los videos o datasets del proyecto. Instalación: `entornos/instalar.md`, secciones 1 a 3.

| Paso | Qué hace | Sale a `runs/herramienta/<video>/` |
|---|---|---|
| 1 | Lee tus videos (o los baja de sus URL) | — |
| 2 | Detector → tracker → unión de fragmentos | `tracks__<detector>.csv`, `trayectoria.csv`, `trayectoria.png` |
| 3 | 7 puntos con dos métodos | `puntos_*.csv`, `*_fondo.png`, `*_trazo.png` |
| 4 | Profundidad relativa en los 7 puntos (Video Depth Anything Small) | `puntos3d_*.csv`, `trayectoria3d_*.html` y `.jpg`, `tira_*.jpg` |

**Funciona mejor con** cámara fija y un ave a la vez, como los videos con los que se ajustó. Con la cámara en movimiento,
la trayectoria 2D se mueve con ella y la máscara de movimiento del paso 4 deja de valer. Si hay varias aves, se conservan
los fragmentos más largos que no se solapan en el tiempo: el resultado es **una** trayectoria.

Como no hay verdad de terreno para tus videos, solo verás estadística descriptiva, no métricas de acierto.
**La 3D es exploratoria:** `z_norm` es relativa a tu escena, no una distancia, y no está validada en general (D-13 a D-18).

Usa solo videos sobre los que tengas derechos. Las salidas quedan fuera de git: no subas figuras con cuadros de video.

In [ ]:
%run 00_comun.ipynb

## 1. Tus videos

- **`CARPETA_VIDEOS`:** la carpeta donde descargaste los archivos, o un video. Por defecto, `datos/mis_videos/` en la zona de
  trabajo. Formatos: mp4, mov, avi, mkv, webm, m4v.
- **`ARCHIVOS`:** opcional, para procesar solo algunos archivos de esa carpeta.
- **`URLS`:** opcional. Se bajan a la carpeta con `yt-dlp` antes de empezar.
- **`DETECTOR_HERRAMIENTA`:** por defecto el vigente de `config.yaml`. Puedes poner otro `best.pt`.

In [ ]:
CARPETA_VIDEOS = P["mis_videos"]     # o Path(r"D:\mis_descargas")
ARCHIVOS = None                      # o ["clip1.mp4", "clip2.mov"]
URLS = []                            # o ["https://..."]
DETECTOR_HERRAMIENTA = DETECTOR      # o Path(r"D:\...\best.pt")

descargar_urls(URLS, CARPETA_VIDEOS)
VIDS = leer_fuente(CARPETA_VIDEOS, ARCHIVOS)
print("detector:", DETECTOR_HERRAMIENTA)
display(pd.DataFrame(VIDS).T.drop(columns="path").rename_axis("video"))

## 2. Trayectorias y 7 puntos

Usa la configuración vigente de `config.yaml`: ByteTrack combo (D-09), unión prudente (D-10) y 7 puntos en total por video
(D-12). En `a_uniforme_tiempo` los puntos van equiespaciados en el tiempo con ave; en `b_salientes` son los que más forma aportan.

**Cómo revisar:** `pct_con_ave` es el porcentaje de frames con ave; con 0 el detector no encontró ninguna. Muchos `segmentos`
indican salidas de cuadro o detecciones perdidas. Un **puente** es un hueco interpolado dentro de un segmento y sale en
discontinua en `trayectoria.png`; si hay `puentes_largos`, revisa que no una dos visitas distintas.

In [ ]:
RESUMEN = pd.DataFrame([procesar_video(n, v, DETECTOR_HERRAMIENTA) for n, v in VIDS.items()])
P["herramienta"].mkdir(parents=True, exist_ok=True); RESUMEN.to_csv(P["herramienta"] / "resumen.csv", index=False)
display(RESUMEN)
from IPython.display import Image as _Img
for n in RESUMEN[RESUMEN.siete_puntos].video:
    display(_Img(filename=str(carpeta_salida(n) / "b_salientes_fondo.png"), width=800))
print("salidas en", P["herramienta"])

## 3. Elegir las regiones de cada video

Para cada punto se lee la profundidad del ave y se normaliza dentro del mismo mapa (D-14):

`z_norm = (z_ave − z_flor) / (z_fondo − z_flor)`: 0 = flor, 1 = fondo, negativo = delante.

Hacen falta dos rectángulos por video, en coordenadas de 0 a 1 (`x0, y0, x1, y1`):

- **flor:** lo que visita el ave;
- **fondo:** una zona lejana y estable por donde no pase.

La celda siguiente muestra el frame mediano de cada video con una rejilla. Escribe tus rectángulos en `REGIONES`, con el nombre
de video de la tabla del paso 1. **Los videos sin regiones se saltan la 3D** y se quedan con su resultado 2D.

In [ ]:
for n in RESUMEN[RESUMEN.siete_puntos].video:
    ver_cuadro(n, VIDS[n])

In [ ]:
REGIONES = {
    # "nombre_del_video": {"flor": ("flor", (0.54, 0.38, 0.69, 0.58)), "fondo": ("bosque", (0.30, 0.03, 0.62, 0.16))},
}

## 4. Profundidad y trayectoria 3D

La primera vez, cada video tarda unos minutos porque el modelo lo procesa completo dos veces, y necesita bastante memoria: en
la laptop de referencia (6 GB de GPU), i5bE, de 601 frames, usó 3,2 GB. Prueba primero con clips cortos. Los mapas quedan en
`runs/herramienta/_cache_vda/` y no se recalculan.

**La resolución de entrada (`config.yaml` → `vda.sizes`, por defecto 392) influye en `z_norm`:** en i5bE, la lectura de un punto cambió
hasta 0,46 entre 448 y 392. Compara solo resultados con la misma resolución; si un video baja de resolución por memoria, lo dice al calcularlo.

In [ ]:
listos = [n for n in RESUMEN[RESUMEN.siete_puntos].video]
sin_regiones = [n for n in listos if n not in REGIONES]
if sin_regiones:
    print("Aviso: sin REGIONES, se salta la 3D de:", sin_regiones)
con_regiones = {n: VIDS[n] for n in listos if n in REGIONES}
CACHE_HERRAMIENTA = P["herramienta"] / "_cache_vda"
if con_regiones:
    mapas_vda({n: v["path"] for n, v in con_regiones.items()}, CACHE_HERRAMIENTA)
    filas = []
    for n, v in con_regiones.items():
        filas += procesar_3d(n, v, REGIONES[n], CACHE_HERRAMIENTA)
    CTRL = pd.DataFrame(filas); CTRL.to_csv(P["herramienta"] / "control_ruido.csv", index=False)
    display(CTRL.round(3))

**Cómo leer el control de ruido.** `forma_supera_ruido = True` significa que la diferencia típica entre puntos es mayor que la
variación entre frames vecinos: la forma en z tiene alguna señal. Con `False`, no la interpretes. Revisa además `tira_*.jpg`, el
recorte de cada punto con su `z_norm`, y abre `trayectoria3d_*.html` para girar la figura.

## ¿Otro detector?

Cambia `DETECTOR_HERRAMIENTA` y vuelve a ejecutar: las detecciones de cada detector se guardan aparte, pero las demás salidas de
la carpeta del video se sustituyen. Los detectores nuevos del proyecto llegan con `git pull` (`modelos/README.md`). Para entrenar el tuyo con
tu dataset, usa los notebooks 01 (sección 2) y 02 (sección 6).